# 16 — Evaluación: métricas, umbral de decisión, bootstrap y eventos

Esta notebook verifica el módulo `src/evaluacion.py`, con el que se evalúan todos los modelos de la tesis (referencias, CNN plana y ResNet, con L = 2, 3 y 4). La evaluación se escribe y se prueba antes que los modelos para que todos se midan con la misma regla.

Las pruebas usan datos sintéticos construidos de modo que la respuesta correcta se conoce de antemano: si una función devuelve otra cosa, la función está mal. Con un modelo real no hay forma de saber cuál es la respuesta correcta y, por lo tanto, no hay forma de detectar un error de cálculo.

La Sección 5 es la única que lee datos reales (las etiquetas) y calcula la duración de los episodios de lluvia fuerte, que respalda el largo del bloque del bootstrap.

## Decisiones de evaluación

| # | Decisión | Motivo |
|-|-|-|
| 1 | Bootstrap con **B = 10,000** réplicas en validación y prueba | El número de réplicas no cambia el ancho del intervalo (eso lo fija la cantidad de días), solo el ruido de simulación en sus extremos. Ese ruido baja con la raíz de B; con 10,000 queda por debajo de la precisión con que se reportan las métricas (dos decimales), y el costo es despreciable porque cada réplica solo suma tablas de contingencia por bloque. Se verifica en la prueba 3.2 con dos semillas. |
| 2 | Unidad de remuestreo: **día pluviométrico** (07:00 a 07:00, hora local America/Bogota) con todas las estaciones, asignado por el instante t de la imagen | Una tormenta y los pares que dependen de ella (historia de hasta 30 min y ventana de etiqueta de 1 a 3 h) deben quedar en un mismo bloque. El corte de las 07:00 es la convención del IDEAM para el total diario de lluvia y, en estos datos, parte solo el 2.3 % de los episodios regionales (medianoche local: 8.6 %; medianoche UTC, 19:00 local: 15.9 %; mínimo observado: 2.0 % a las 08:00–09:00). Respaldo empírico en la Sección 5; la prueba de sensibilidad con bloques de 6, 12, 24 y 48 h se hace sobre las predicciones de la primera referencia (S5). |
| 3 | Comparación de modelos por **diferencias pareadas** | Los modelos se evalúan sobre los mismos pares (conjunto común L_max = 4); los días fáciles y difíciles afectan a todos por igual, y ese efecto se cancela al restar dentro de cada réplica. Comparaciones fijadas antes de abrir prueba: CNN frente a referencia, ResNet frente a CNN plana, L = 4 frente a L = 2 y L = 3 frente a L = 2. |
| 4 | **Umbral de decisión** por CSI máximo en validación, grilla 0.01–0.99 con paso 0.01, congelado para prueba; desempate por el **punto medio de la meseta** | Las probabilidades salen infladas por el muestreo de entrenamiento (1 positivo por cada 4 negativos frente a ~1:290 real), así que 0.5 no tiene un significado especial. La grilla evita perseguir el ruido de un valor exacto; el punto medio de la meseta queda lejos de los bordes donde el CSI empieza a caer. En el bootstrap el umbral queda fijo; como diagnóstico se observa la dispersión del umbral re-elegido en cada réplica. |
| 5 | **ETS** (Gilbert) junto al CSI | Comprometido en la metodología y estándar en la literatura. Con la prevalencia de este conjunto los aciertos por azar son casi nulos y el ETS queda muy cerca del CSI; la corrección pesa en los estratos con prevalencias distintas. |
| 6 | Denominador cero → **NaN**, nunca 0 | Un FAR = 0 sin alertas o un POD = 0 sin tormentas dirían algo falso del modelo. En el bootstrap se cuentan las réplicas indefinidas; si superan el 1 % (regla práctica fijada antes de ver resultados), el intervalo se declara poco confiable. |
| 7 | **Pruebas sintéticas** de esta notebook | Cada prueba verifica una decisión de esta tabla o un error que ya ocurrió en el proyecto. |
| 8 | **Salida** en nombres planos (`val_csi`, `val_csi_ic_inf`, …) para MLflow, con tablas por bloque y réplicas como artefactos | Ningún resultado de la tesis debe depender de volver a entrenar o a predecir. |

Convención: una alerta se emite cuando la probabilidad es mayor o igual que el umbral. En la tabla de contingencia, a = aciertos, b = falsas alarmas, c = sorpresas (tormentas sin alerta) y d = rechazos correctos.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "modelo" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import evaluacion as ev

def verificar(condicion, mensaje):
    assert condicion, "FALLA: " + mensaje
    print("OK  ", mensaje)

### Generador de datos sintéticos realistas

Varias pruebas necesitan un conjunto con la estructura del real: días pluviométricos completos (de 07:00 a 07:00 local), 61 estaciones (las del modelado, sin Nueva Generación), un par cada 10 minutos, prevalencia baja y tormentas que afectan a varias estaciones a la vez. El generador asigna a cada día una propensión a tormenta compartida por todas las estaciones, de modo que los pares de un mismo día están correlacionados como en los datos reales. Las probabilidades del "modelo" dependen de la etiqueta más ruido, para que haya aciertos, falsas alarmas y sorpresas.

In [ ]:
def conjunto_sintetico(n_dias=47, n_estaciones=61, inicio="2024-11-15", prevalencia=0.005, semilla=0):
    rng = np.random.default_rng(semilla)
    t_local = pd.date_range(pd.Timestamp(inicio) + pd.Timedelta(hours=ev.HORA_INICIO_BLOQUE), periods=n_dias * 144, freq="10min", tz=ev.ZONA_LOCAL)
    t = np.repeat(t_local.tz_convert("UTC"), n_estaciones)
    estacion = np.tile(np.arange(n_estaciones), len(t_local))
    dia = np.repeat(np.arange(n_dias), 144 * n_estaciones)
    propension = rng.gamma(0.4, 1.0, n_dias)                 # días secos y días muy lluviosos
    prob = np.clip(prevalencia * propension[dia] / propension.mean(), 0, 1)
    y = (rng.random(len(t)) < prob).astype(int)
    logit = -4.5 + 3.0 * y + 1.0 * np.log1p(propension[dia]) + rng.normal(0, 1.2, len(t))
    p = 1 / (1 + np.exp(-logit))
    return pd.DataFrame({"timestamp": t, "estacion": estacion, "y": y, "p": p})

val = conjunto_sintetico()
print(f"{len(val):,} pares, {val.y.sum():,} positivos ({val.y.mean():.3%}), "
      f"{len(np.unique(ev.bloque_local(val.timestamp)))} días pluviométricos")

412,848 pares, 2,063 positivos (0.500%), 47 días pluviométricos


## 1. Métricas de la tabla de contingencia

### 1.1 Tabla calculada a mano

Caso pequeño con a = 6, b = 4, c = 2, d = 88 (n = 100). A mano: POD = 6/8 = 0.75; FAR = 4/10 = 0.4; CSI = 6/12 = 0.5. Para el ETS, los aciertos por azar son a_r = (a + b)(a + c)/n = 10·8/100 = 0.8, y ETS = (6 − 0.8)/(12 − 0.8) = 5.2/11.2 = 0.46429. Si esta prueba falla, la fórmula implementada no es la de Gilbert.

In [ ]:
y = np.r_[np.ones(6), np.zeros(4), np.ones(2), np.zeros(88)]
p = np.r_[np.ones(6), np.ones(4), np.zeros(2), np.zeros(88)]
tabla = ev.tabla_contingencia(y, p, 0.5)
m = ev.metricas(*tabla)
verificar(tabla == (6, 4, 2, 88), f"tabla (a, b, c, d) = {tabla}")
verificar(np.isclose(m["pod"], 0.75), f"POD = {m['pod']:.5f}")
verificar(np.isclose(m["far"], 0.4), f"FAR = {m['far']:.5f}")
verificar(np.isclose(m["csi"], 0.5), f"CSI = {m['csi']:.5f}")
verificar(np.isclose(m["ets"], 5.2 / 11.2), f"ETS = {m['ets']:.5f}")

OK   tabla (a, b, c, d) = (6, 4, 2, 88)
OK   POD = 0.75000
OK   FAR = 0.40000
OK   CSI = 0.50000
OK   ETS = 0.46429


### 1.2 Modelos extremos

- **Perfecto** (alerta exactamente cuando hay tormenta): POD = 1, FAR = 0, CSI = 1, ETS = 1.
- **Nunca alerta**: POD = 0, CSI = 0, FAR = NaN (sin alertas no hay falsas alarmas que contar; un 0 diría que el modelo es preciso cuando no dijo nada).
- **Siempre alerta**: POD = 1, FAR = 1 − prevalencia, CSI = prevalencia.

Se usa el conjunto sintético de validación, con su prevalencia real de ~0.5 %.

In [ ]:
y, n = val.y.to_numpy(), len(val)
prev = y.mean()

m = ev.metricas(*ev.tabla_contingencia(y, y.astype(float), 0.5))
verificar(m == {"pod": 1.0, "far": 0.0, "csi": 1.0, "ets": 1.0}, f"perfecto: {m}")

m = ev.metricas(*ev.tabla_contingencia(y, np.zeros(n), 0.5))
verificar(m["pod"] == 0 and m["csi"] == 0 and np.isnan(m["far"]), f"nunca alerta: {m}")

m = ev.metricas(*ev.tabla_contingencia(y, np.ones(n), 0.5))
verificar(m["pod"] == 1 and np.isclose(m["far"], 1 - prev) and np.isclose(m["csi"], prev),
          f"siempre alerta: POD = {m['pod']}, FAR = {m['far']:.5f} (1 − prev = {1 - prev:.5f}), CSI = {m['csi']:.5f}")

OK   perfecto: {'pod': 1.0, 'far': 0.0, 'csi': 1.0, 'ets': 1.0}
OK   nunca alerta: {'pod': 0.0, 'far': nan, 'csi': 0.0, 'ets': 0.0}
OK   siempre alerta: POD = 1.0, FAR = 0.99500 (1 − prev = 0.99500), CSI = 0.00500


### 1.3 Modelo al azar

Un modelo que alerta al azar con la misma frecuencia que las tormentas (prevalencia π) tiene, en esperanza, a = nπ², b = c = nπ(1 − π), de modo que CSI = π/(2 − π) y ETS = 0. Esta prueba confirma que el ETS descuenta los aciertos por suerte. Como el resultado es aleatorio, se verifica dentro de un margen: se usan n = 2,000,000 pares y π = 0.05 (y no la prevalencia real) para que el error de muestreo sea pequeño frente al margen del 5 % relativo en el CSI y de ±0.005 en el ETS.

In [ ]:
rng = np.random.default_rng(1)
pi_, n_azar = 0.05, 2_000_000
y_az = (rng.random(n_azar) < pi_).astype(int)
p_az = (rng.random(n_azar) < pi_).astype(float)
m = ev.metricas(*ev.tabla_contingencia(y_az, p_az, 0.5))
esperado = pi_ / (2 - pi_)
verificar(abs(m["csi"] / esperado - 1) < 0.05, f"azar: CSI = {m['csi']:.5f}, esperado π/(2 − π) = {esperado:.5f}")
verificar(abs(m["ets"]) < 0.005, f"azar: ETS = {m['ets']:.5f} (esperado ≈ 0)")

OK   azar: CSI = 0.02581, esperado π/(2 − π) = 0.02564
OK   azar: ETS = 0.00012 (esperado ≈ 0)


### 1.4 Etiquetas o probabilidades NaN

Las etiquetas NaN (vacíos de registro de las estaciones) se quitan antes de evaluar: el índice de `11-dataset_training` y el cargador solo entregan pares con etiqueta. Si un NaN llega a la evaluación, algo falló aguas arriba. Por eso la función se detiene con un error en lugar de descartarlo en silencio, que escondería el problema.

In [ ]:
for nombre, y_, p_ in [("etiqueta NaN", [1, 0, np.nan, 1], [0.8, 0.1, 0.9, 0.2]),
                       ("probabilidad NaN", [1, 0, 0, 1], [0.8, np.nan, 0.9, 0.2])]:
    try:
        ev.tabla_contingencia(y_, p_, 0.5)
        verificar(False, f"{nombre}: debía detenerse")
    except ValueError as e:
        verificar(True, f"{nombre}: se detiene ({e})")

OK   etiqueta NaN: se detiene (Hay 1 etiquetas NaN. Las etiquetas NaN deben quitarse antes de evaluar; si llegan aquí, algo falló aguas arriba.)
OK   probabilidad NaN: se detiene (Hay 1 probabilidades NaN.)


### 1.5 Conjunto sin tormentas

Pasa en réplicas del bootstrap que sortean solo días secos, en estratos pequeños de S4b y en estaciones con pocas tormentas. Sin tormentas, POD = 0/0 = NaN; el CSI vale 0 si hubo falsas alarmas (el modelo solo se equivocó) y NaN si tampoco hubo alertas. Nada debe interrumpir la ejecución.

In [ ]:
y0 = np.zeros(1000)
m = ev.metricas(*ev.tabla_contingencia(y0, np.r_[np.ones(5), np.zeros(995)], 0.5))
verificar(np.isnan(m["pod"]) and m["far"] == 1 and m["csi"] == 0, f"sin tormentas, con 5 alertas: {m}")
m = ev.metricas(*ev.tabla_contingencia(y0, np.zeros(1000), 0.5))
verificar(all(np.isnan(v) for v in m.values()), f"sin tormentas ni alertas: {m}")

OK   sin tormentas, con 5 alertas: {'pod': nan, 'far': 1.0, 'csi': 0.0, 'ets': 0.0}
OK   sin tormentas ni alertas: {'pod': nan, 'far': nan, 'csi': nan, 'ets': nan}


## 2. Umbral de decisión

### 2.1 Separación conocida

Todas las tormentas tienen probabilidad 0.70 y todos los demás pares 0.30. Cualquier umbral en (0.30, 0.70] separa perfectamente, así que en la grilla la meseta de CSI = 1 va de 0.31 a 0.70 y su punto medio es 0.50 (con un número par de puntos en la meseta se toma el más bajo de los dos centrales, regla fija del módulo).

In [ ]:
r = ev.elegir_umbral(y, np.where(y == 1, 0.70, 0.30))
verificar(r["csi_max"] == 1.0, f"CSI máximo = {r['csi_max']}")
verificar((r["meseta_desde"], r["meseta_hasta"]) == (0.31, 0.70), f"meseta = [{r['meseta_desde']}, {r['meseta_hasta']}]")
verificar(r["umbral"] == 0.50, f"umbral elegido = {r['umbral']}")

OK   CSI máximo = 1.0
OK   meseta = [0.31, 0.7]
OK   umbral elegido = 0.5


### 2.2 Meseta con máximo compartido

Con probabilidades agrupadas, varios umbrales seguidos dan exactamente el mismo CSI máximo. Aquí las tormentas tienen probabilidad 0.62, un grupo de negativos 0.40 y el resto 0.10: entre 0.41 y 0.62 el CSI es el mismo. El umbral elegido debe ser el punto medio (0.51), no el primero ni el último, y debe ser el mismo en cada ejecución. Se verifica además que la búsqueda vectorizada coincide con calcular el CSI umbral por umbral.

In [ ]:
rng = np.random.default_rng(2)
y_m = np.r_[np.ones(50), np.zeros(950)]
p_m = np.r_[np.full(40, 0.62), np.full(10, 0.05), np.full(30, 0.40), np.full(920, 0.10)]
r = ev.elegir_umbral(y_m, p_m)
verificar((r["meseta_desde"], r["meseta_hasta"]) == (0.41, 0.62), f"meseta = [{r['meseta_desde']}, {r['meseta_hasta']}]")
verificar(r["umbral"] == 0.51, f"umbral elegido = {r['umbral']} (punto medio)")
directo = [ev.metricas(*ev.tabla_contingencia(y_m, p_m, u))["csi"] for u in ev.GRILLA_UMBRALES]
verificar(np.allclose(directo, r["curva"]["csi"], equal_nan=True), "búsqueda vectorizada = cálculo umbral por umbral")

OK   meseta = [0.41, 0.62]
OK   umbral elegido = 0.51 (punto medio)
OK   búsqueda vectorizada = cálculo umbral por umbral


## 3. Bootstrap por bloques

### 3.1 Asignación de bloques en hora local

El instante t se asigna a su día pluviométrico local (07:00 a 07:00), no a su día UTC. Un par a las 11:50 UTC del 20-nov es 06:50 local y pertenece al día pluviométrico que empezó el 19-nov a las 07:00; uno a las 12:00 UTC es 07:00 local y abre el bloque del 20-nov. Un par a las 04:30 UTC del 20-nov (23:30 local del 19) cae en el bloque del 19, aunque en UTC ya sea otro día. Timestamps sin zona horaria detienen la función: mezclar UTC con y sin zona ya produjo cruces vacíos en silencio en este proyecto. Con bloques de 12 h, los cortes son a las 07:00 y a las 19:00 locales.

In [ ]:
ts = pd.to_datetime(["2024-11-20 04:30", "2024-11-20 11:50", "2024-11-20 12:00"]).tz_localize("UTC")
b24 = ev.bloque_local(ts, 24)
dia_19 = (pd.Timestamp("2024-11-19") - pd.Timestamp("1970-01-01")).days
verificar(list(b24 - dia_19) == [0, 0, 1], f"bloques de 24 h: 23:30 y 06:50 locales en el día del 19-nov, 07:00 en el del 20 ({list(b24 - dia_19)})")
b12 = ev.bloque_local(pd.to_datetime(["2024-11-20 23:50", "2024-11-21 00:00"]).tz_localize("UTC"), 12)
verificar(b12[1] == b12[0] + 1, "bloques de 12 h: corte a las 19:00 locales (00:00 UTC)")
try:
    ev.bloque_local(pd.to_datetime(["2024-11-20 04:30"]), 24)
    verificar(False, "timestamps sin zona: debía detenerse")
except ValueError:
    verificar(True, "timestamps sin zona: se detiene")

OK   bloques de 24 h: 23:30 y 06:50 locales en el día del 19-nov, 07:00 en el del 20 ([np.int64(0), np.int64(0), np.int64(1)])
OK   bloques de 12 h: corte a las 19:00 locales (00:00 UTC)
OK   timestamps sin zona: se detiene


### 3.2 Integridad de los bloques con varias estaciones

Cada día pluviométrico debe contener todos sus pares de todas las estaciones: 61 estaciones × 144 instantes = 8,784 pares por día. Además, la tabla del bloque debe ser idéntica a la que se obtiene calculando directamente sobre los pares de ese día.

In [ ]:
bloques, tablas = ev.tablas_por_bloque(val.timestamp, val.y, val.p, 0.3)
verificar((tablas.sum(1) == 61 * 144).all(), f"{len(bloques)} bloques, todos con {61 * 144:,} pares")
dia = ev.bloque_local(val.timestamp)
sel = dia == dia[0]
verificar(tuple(tablas[0]) == ev.tabla_contingencia(val.y[sel], val.p[sel], 0.3), "tabla del primer día = cálculo directo sobre sus pares")

OK   47 bloques, todos con 8,784 pares
OK   tabla del primer día = cálculo directo sobre sus pares


### 3.3 Equivalencia del atajo de cálculo

El bootstrap no reconstruye los pares de cada réplica: suma las tablas de los días sorteados, ponderadas por cuántas veces salió cada día. La prueba reconstruye explícitamente los pares de tres réplicas (concatenando los días sorteados, con repeticiones) y verifica que las métricas coinciden con las del atajo. Si fallara, todos los intervalos estarían mal aunque el resto de las pruebas pasara.

In [ ]:
pesos = ev.pesos_bootstrap(len(bloques), B=3, semilla=7)
atajo = ev.metricas(*(pesos @ tablas).T)
id_bloque = ev.bloque_local(val.timestamp, 24)
for r_ in range(3):
    idx = np.concatenate([np.flatnonzero(id_bloque == b) for b, k in zip(bloques, pesos[r_]) for _ in range(k)])
    directo = ev.metricas(*ev.tabla_contingencia(val.y.to_numpy()[idx], val.p.to_numpy()[idx], 0.3))
    verificar(all(np.isclose(directo[k], atajo[k][r_]) for k in ev.METRICAS), f"réplica {r_}: atajo = reconstrucción explícita")

OK   réplica 0: atajo = reconstrucción explícita
OK   réplica 1: atajo = reconstrucción explícita
OK   réplica 2: atajo = reconstrucción explícita


### 3.4 Modelo perfecto y modelo comparado consigo mismo

Un modelo perfecto debe dar CSI = 1 en las 10,000 réplicas, con intervalo [1, 1]. La diferencia pareada de un modelo consigo mismo debe ser exactamente 0 en todas las réplicas; si no lo fuera, los días no se estarían sorteando igual para los dos modelos.

In [ ]:
pesos = ev.pesos_bootstrap(len(bloques), semilla=42)
_, t_perf = ev.tablas_por_bloque(val.timestamp, val.y, val.y.astype(float), 0.5)
bs = ev.bootstrap(t_perf, pesos)
verificar(bs.ic["csi"] == (1.0, 1.0), f"modelo perfecto: IC del CSI = {bs.ic['csi']}")

comp = ev.comparar_pareado(tablas, tablas, pesos)
verificar(np.all(comp["replicas"] == 0), "modelo frente a sí mismo: Δ = 0 en las 10,000 réplicas")

OK   modelo perfecto: IC del CSI = (1.0, 1.0)
OK   modelo frente a sí mismo: Δ = 0 en las 10,000 réplicas


### 3.5 Estabilidad frente a la semilla (decisión 1)

Con B = 10,000 el resultado no debe depender de la semilla del sorteo. Se corre el bootstrap con dos semillas y se exige que los extremos de cada intervalo difieran en menos de 0.005, media unidad del segundo decimal, que es la precisión con que se reportan las métricas. Se informa la diferencia máxima observada.

In [ ]:
r1 = ev.bootstrap(tablas, ev.pesos_bootstrap(len(bloques), semilla=42))
r2 = ev.bootstrap(tablas, ev.pesos_bootstrap(len(bloques), semilla=2026))
dif = max(abs(np.array(r1.ic[k]) - np.array(r2.ic[k])).max() for k in ev.METRICAS)
for k in ev.METRICAS:
    print(f"   {k.upper():4s} semilla 42: [{r1.ic[k][0]:.4f}, {r1.ic[k][1]:.4f}]   semilla 2026: [{r2.ic[k][0]:.4f}, {r2.ic[k][1]:.4f}]")
verificar(dif < 0.005, f"diferencia máxima entre semillas = {dif:.4f}")

   POD  semilla 42: [0.4280, 0.6553]   semilla 2026: [0.4304, 0.6557]
   FAR  semilla 42: [0.5493, 0.6387]   semilla 2026: [0.5494, 0.6382]
   CSI  semilla 42: [0.2455, 0.3613]   semilla 2026: [0.2464, 0.3609]
   ETS  semilla 42: [0.2439, 0.3575]   semilla 2026: [0.2448, 0.3572]
OK   diferencia máxima entre semillas = 0.0025


### 3.6 Comparación pareada y conteo de réplicas indefinidas

Un segundo "modelo" con más ruido debe salir peor que el primero, con un intervalo de la diferencia que no incluya el 0. Se informa también la fracción de réplicas en que gana cada modelo y cuántas réplicas quedaron indefinidas en cada métrica (decisión 6).

In [ ]:
rng = np.random.default_rng(3)
p_peor = np.clip(val.p + rng.normal(0, 0.15, len(val)), 0, 1)
_, t_peor = ev.tablas_por_bloque(val.timestamp, val.y, p_peor, 0.3, bloques=bloques)
comp = ev.comparar_pareado(tablas, t_peor, pesos)
print(f"   ΔCSI = {comp['delta']:.4f}, IC 95 % [{comp['ic_inf']:.4f}, {comp['ic_sup']:.4f}], gana el primero en {comp['frac_gana_A']:.1%} de las réplicas")
verificar(comp["ic_inf"] > 0, "el modelo más ruidoso sale peor y el IC de Δ excluye el 0")
print("   réplicas indefinidas:", r1.n_nan, "| IC poco confiable:", {k: v for k, v in r1.poco_confiable.items() if v} or "ninguno")

   ΔCSI = 0.2531, IC 95 % [0.2099, 0.2729], gana el primero en 100.0% de las réplicas
OK   el modelo más ruidoso sale peor y el IC de Δ excluye el 0
   réplicas indefinidas: {'pod': 0, 'far': 0, 'csi': 0, 'ets': 0} | IC poco confiable: ninguno


### 3.7 Diagnóstico de estabilidad del umbral (opción B de la decisión 4)

En la evaluación el umbral queda fijo. Como diagnóstico se re-elige el umbral en cada réplica y se observa su dispersión: si salta en un rango amplio, la elección del umbral es inestable y hay que declararlo. Aquí solo se verifica que la función corre y devuelve umbrales de la grilla; su interpretación se hace con las predicciones reales a partir de S5.

In [ ]:
bloques_u, t_bu = ev.tablas_por_bloque_y_umbral(val.timestamp, val.y, val.p)
u_rep = ev.estabilidad_umbral(t_bu, ev.pesos_bootstrap(len(bloques_u), B=1000, semilla=42))
q = np.quantile(u_rep, [0.025, 0.5, 0.975])
print(f"   umbral re-elegido en 1,000 réplicas: mediana {q[1]:.2f}, 95 % central [{q[0]:.2f}, {q[2]:.2f}]")
verificar(np.isin(u_rep, ev.GRILLA_UMBRALES).all(), "todos los umbrales re-elegidos pertenecen a la grilla")

   umbral re-elegido en 1,000 réplicas: mediana 0.36, 95 % central [0.34, 0.41]
OK   todos los umbrales re-elegidos pertenecen a la grilla


### 3.8 Tiempo con el tamaño del conjunto de prueba

El conjunto de prueba tiene 2,395,490 pares en 365 días. Se mide el tiempo de armar las tablas por día y de las 10,000 réplicas con un conjunto sintético de ese tamaño (6,563 pares por día). El cuello de botella posible era recorrer millones de filas en cada réplica; con el atajo de tablas por bloque, el costo de las réplicas no depende del número de pares.

In [ ]:
grande = conjunto_sintetico(n_dias=365, n_estaciones=46, inicio="2025-01-01", semilla=5)  # 365 × 144 × 46 ≈ 2.42 M pares
print(f"   {len(grande):,} pares")
t0 = time.perf_counter(); bl, tb = ev.tablas_por_bloque(grande.timestamp, grande.y, grande.p, 0.3); t1 = time.perf_counter()
_ = ev.bootstrap(tb, ev.pesos_bootstrap(len(bl), semilla=42)); t2 = time.perf_counter()
print(f"   tablas por día: {t1 - t0:.1f} s; 10,000 réplicas: {t2 - t1:.2f} s")
verificar(t2 - t1 < 10, "las 10,000 réplicas tardan menos de 10 s")

   2,417,760 pares
   tablas por día: 0.2 s; 10,000 réplicas: 0.28 s
OK   las 10,000 réplicas tardan menos de 10 s


## 4. Salida para MLflow

La función `evaluar` reúne todo para un conjunto y un umbral ya fijado: métricas puntuales, intervalos, conteos (pares, tormentas, alertas, bloques y bloques con tormenta) y réplicas indefinidas, con nombres planos listos para `mlflow.log_metrics`. Las tablas por bloque y las réplicas quedan como artefactos: con las tablas se puede recalcular cualquier intervalo sin volver a correr el modelo.

In [ ]:
planas, artefactos = ev.evaluar(val.timestamp, val.y, val.p, umbral=0.3, conjunto="val")
for k, v in planas.items():
    print(f"   {k:28s} {v}")
verificar(all(isinstance(k, str) and k.startswith("val_") for k in planas), "todas las métricas con prefijo del conjunto")
verificar(artefactos["tablas_por_bloque"].to_numpy().sum() == len(val), "las tablas por bloque cubren todos los pares")

   val_umbral                   0.3
   val_pod                      0.5521085797382452
   val_pod_ic_inf               0.42797667565202463
   val_pod_ic_sup               0.6553317114813836
   val_replicas_nan_pod         0
   val_far                      0.5781481481481482
   val_far_ic_inf               0.5492894150981582
   val_far_ic_sup               0.638674485890604
   val_replicas_nan_far         0
   val_csi                      0.3142935982339956
   val_csi_ic_inf               0.24546399537964347
   val_csi_ic_sup               0.36125020904252
   val_replicas_nan_csi         0
   val_ets                      0.31173122324633407
   val_ets_ic_inf               0.24388607631020623
   val_ets_ic_sup               0.35745631681121814
   val_replicas_nan_ets         0
   val_n_pares                  412848
   val_n_tormentas              2063
   val_n_alertas                2700
   val_n_bloques                47
   val_n_bloques_con_tormenta   39
   val_horas_bloque            

## 5. Duración de los episodios de lluvia fuerte (datos reales)

Respaldo físico del largo de bloque (decisión 2). Una racha es una sucesión de pares positivos consecutivos (sin saltos de más de 10 min): todos esos pares dependen de la misma tormenta y por eso deberían caer en un mismo bloque. Se calculan rachas por estación y regionales (instantes con al menos una estación positiva), sobre validación y prueba, sin Nueva Generación (excluida del modelado). La duración de una racha incluye la anticipación de la etiqueta: un positivo en t anuncia lluvia en (t+1 h, t+3 h], así que la racha empieza hasta 3 h antes que la lluvia.

Si la mayoría de las rachas regionales dura bastante menos de 24 h y pocas cruzan el corte del bloque, el bloque de 24 h contiene la dependencia. La prueba de sensibilidad con bloques de 6, 12, 24 y 48 h se corre en S5, con las primeras predicciones reales.

Esta sección requiere `etiquetas_ground_truth.parquet` (solo en la máquina de trabajo). Antes de usar las columnas se imprimen sus nombres, tipos y zona horaria.

In [ ]:
etiquetas = pd.read_parquet(RAIZ / "data_ingestion" / "ground_truth" / "etiquetas_ground_truth.parquet")
print(etiquetas.dtypes)
print("zona horaria de timestamp:", etiquetas["timestamp"].dt.tz)

codigoestacion                     object
timestamp             datetime64[ns, UTC]
etiqueta                          float64
en_curso                             bool
inminente                            bool
cobertura                            bool
sin_dato_previo                      bool
hueco_primera_hora                   bool
int_t                             float64
dtype: object
zona horaria de timestamp: UTC


In [ ]:
COL_ESTACION = "codigoestacion"
NUEVA_GENERACION = "0021206600"
INICIO_VAL = pd.Timestamp("2024-11-15", tz=ev.ZONA_LOCAL)

e = etiquetas.dropna(subset=["etiqueta"])
verificar((e[COL_ESTACION].astype(str) == NUEVA_GENERACION).any(), f"Nueva Generación ({NUEVA_GENERACION}) presente antes de excluirla")
e = e[(e[COL_ESTACION].astype(str) != NUEVA_GENERACION) & ~e["en_curso"].astype(bool) & ~e["inminente"].astype(bool)]
e = e[e["timestamp"] >= INICIO_VAL]
rachas = ev.rachas_positivas(e, col_estacion=COL_ESTACION)

inicio_local = rachas["inicio"].dt.tz_convert(ev.ZONA_LOCAL)
fin_local = rachas["fin"].dt.tz_convert(ev.ZONA_LOCAL)
corte = pd.Timedelta(hours=ev.HORA_INICIO_BLOQUE)
rachas["cruza_corte"] = (inicio_local - corte).dt.date != (fin_local - corte).dt.date
rachas["tipo"] = np.where(rachas["estacion"] == "red", "regional", "por estación")

resumen = rachas.groupby("tipo").agg(
    n=("duracion_h", "size"),
    mediana_h=("duracion_h", "median"),
    p90_h=("duracion_h", lambda s: s.quantile(0.9)),
    max_h=("duracion_h", "max"),
    frac_cruza_corte_7h=("cruza_corte", "mean"),
)
resumen

OK   Nueva Generación (0021206600) presente antes de excluirla


,n,mediana_h,p90_h,max_h,frac_cruza_corte_7h
tipo,,,,,
por estación,1264,1.833333,2.333333,3.000000,0.009494
regional,301,2.000000,3.833333,12.833333,0.023256


### 5.1 Hora de corte del bloque

Fracción de rachas regionales que quedarían partidas en dos bloques según la hora local a la que empiece el bloque de 24 h. La serie mide la estructura temporal de las tormentas en validación y prueba, no el desempeño de ningún modelo, así que usarla para fijar el corte no filtra información del resultado.

Resultado (12-oct-2026): el mínimo está entre las 08:00 y las 09:00 (2.0 %); las 07:00 parten el 2.3 %; la medianoche local, el 8.6 %; las 19:00 locales (medianoche UTC), el 15.9 %; el peor corte es a las 13:00–14:00 (~19 %), en el pico convectivo de la tarde. Con el régimen nocturno de validación y prueba, las tormentas de la noche siguen pasada la medianoche y la calma llega al amanecer.

Se adopta el **día pluviométrico del IDEAM, de 07:00 a 07:00**: es la convención con la que la red de estaciones totaliza la lluvia diaria (IDEAM, respuesta al radicado 20239050091784, p. 3, "Día pluviométrico") y queda a 0.3 puntos del mínimo, alrededor de un episodio de 301.

In [ ]:
reg = rachas[rachas.estacion == "red"]
ini_h, fin_h = reg["inicio"].dt.tz_convert(ev.ZONA_LOCAL), reg["fin"].dt.tz_convert(ev.ZONA_LOCAL)
frac_partidos = pd.Series(
    {h: ((ini_h - pd.Timedelta(hours=h)).dt.date != (fin_h - pd.Timedelta(hours=h)).dt.date).mean() for h in range(24)},
    name="frac_partidos").rename_axis("hora_corte_local")
frac_partidos.round(3)

hora_corte_local
0     0.086
1     0.073
2     0.066
3     0.037
4     0.027
5     0.037
6     0.030
7     0.023
8     0.020
9     0.020
10    0.040
11    0.090
12    0.163
13    0.189
14    0.186
15    0.159
16    0.153
17    0.153
18    0.163
19    0.159
20    0.150
21    0.120
22    0.116
23    0.090
Name: frac_partidos, dtype: float64

# Parte II — Métricas por evento y estratos (S4b)

## Decisiones

| # | Decisión | Motivo |
|-|-|-|
| 1 | **Episodio**: por estación, instantes de 10 min con intensidad horaria (`int_t`) mayor o igual al umbral; dos tramos se unen si la intensidad queda bajo el umbral por 30 min o menos. **Inicio**: primer instante del episodio. Solo cuentan los episodios con al menos un par positivo en el conjunto evaluado; los demás se reportan aparte | Las métricas por pares cuentan cada par de 10 min, pero una tormenta genera hasta ~18 pares positivos: alertar en 2 de ellos da POD = 0.11 por pares aunque la tormenta se haya avisado. El episodio usa la misma intensidad que la etiqueta. La tolerancia evita contar dos veces una tormenta que oscila alrededor del umbral. Un episodio sin pares positivos en el conjunto (faltan imágenes o cobertura) no se le pudo avisar al modelo y no debe castigarlo |
| 2 | **Episodio avisado**: al menos una alerta en uno de sus pares positivos con t anterior al inicio. Cada par positivo pertenece al episodio cuyo primer instante fuerte cae en su ventana τ ∈ (t + 2 h, t + 3 h]; si toca dos episodios, al primero | Para quien recibe el aviso, una alerta basta. El vínculo alerta–episodio es la etiqueta misma, sin reglas nuevas de ventana. Una alerta posterior al inicio (en una pausa dentro de la tormenta) no anticipa nada |
| 3 | **Aviso**: alertas de una estación con pausas de 30 min o menos. Se clasifica como **correcto** (contiene una alerta válida), **tardío** (alertas solo en pares positivos posteriores al inicio) o **falso** (ninguna alerta en un par positivo). FAR por evento = falsos / total | Varias alertas seguidas son un solo aviso para quien lo recibe; contarlas por separado castigaría la insistencia, no el error. La misma tolerancia que los episodios. Los tardíos no son falsas alarmas, pero si son muchos indican que el modelo reacciona en lugar de anticipar. Los casi aciertos (lluvia bajo el umbral) cuentan como falsos y se analizan en S12 |
| 4 | **Anticipación** = inicio − t de la primera alerta válida; distribución con mediana, cuartiles, fracción con ≥ 1 h e histograma de 10 min | Medir desde el inicio del aviso inflaría la anticipación cuando el aviso empezó antes de que la tormenta entrara en la ventana. La fracción con ≥ 1 h es el margen que promete el diseño de la etiqueta. El inicio es el fin de la primera hora que alcanza el umbral, la misma convención de la etiqueta |
| 5 | **Estratos**: noche (19:00–06:59 local), terciles de umbral (cortes del CSV de umbrales, fijado con 2024), temporada (DEF, MAM, JJA, SON) y **nube fría** (p10 de C13 ≤ 228.2 K, criterio fijado con train en S3a). Pares por su t; episodios por su inicio (noche, temporada) o por su primer par positivo (nube fría); avisos por su primera alerta. Sin estrato por satélite | La noche es la misma definición del EDA, donde validación y prueba tienen muchos más positivos nocturnos que entrenamiento. Los terciles responden al cambio de 2025, cuando las estaciones de umbral bajo superan mucho más su umbral. La nube fría separa las escenas con convección ya visible de las que no, y conecta con la pregunta de si la red aporta más que "hay nubes frías cerca"; en episodios se clasifica en un momento fijo, igual para todos los modelos, para no sesgar el estrato con el resultado. El satélite no se estratifica: la intercalibración mostró entrada equivalente (≤ 0.22 K, grilla idéntica) y está mezclado con la temporada |
| 6 | **Por estación**, solo en prueba, con un mínimo de **10 episodios evaluables** | Con 10 episodios cada acierto mueve el POD 0.10; con menos, un solo episodio pesa más que las diferencias esperadas entre modelos. Regla práctica fijada antes de ver resultados. Las estaciones por debajo se reportan con sus conteos, sin métricas, y marcadas como insuficientes |

## 6. Episodios y eventos con un caso armado a mano

Dos estaciones durante un día pluviométrico (20-nov, 07:00 a 07:00 local), umbral 20 mm/h, intensidad 25 mm/h en los instantes fuertes y 5 mm/h en el resto. Horas locales:

- **S1**: fuerte a las 15:10, 15:20 y 15:40 (pausa de 10 min a las 15:30, se une) → episodio E1 de 15:10 a 15:40. Fuerte otra vez a las 16:30, tras 40 min bajo el umbral → episodio E2.
- **S2**: fuerte a las 14:00, 14:40, 15:20, 16:00 y 16:40, con pausas de 30 min → un solo episodio E3 de 14:00 a 16:40.

Las etiquetas se calculan aparte, directamente con la regla de la ventana (algún instante fuerte en (t + 2 h, t + 3 h]), y se excluyen los pares con lluvia fuerte en t. Alertas del "modelo": S1 a las 09:00 y 09:20 (aviso falso); S1 a las 11:50, 12:10 y 12:30 (aviso que empieza fuera de la ventana de E1 y entra en ella a las 12:10); S2 a las 14:10 (dentro de E3, después de su inicio: aviso tardío).

Resultados esperados, calculados a mano:
- 3 episodios. El par de S1 a las 13:30 tiene en su ventana (15:30, 16:30] instantes de E1 (15:40) y de E2 (16:30): se asigna a E1, el primero.
- E1 avisado, con primera alerta válida a las 12:10 y anticipación de 3.0 h (no 3 h 20 min desde las 11:50, cuya ventana no contiene lluvia fuerte). E2 no avisado. E3 no avisado (su única alerta es posterior al inicio). POD por evento = 1/3.
- Avisos: falso (09:00–09:20), correcto (11:50–12:30), tardío (14:10). FAR por evento = 1/3.

In [ ]:
def caso_a_mano():
    loc = lambda h: pd.Timestamp(f"2024-11-20 {h}", tz=ev.ZONA_LOCAL).tz_convert("UTC")
    grilla = pd.date_range(pd.Timestamp("2024-11-20 07:00", tz=ev.ZONA_LOCAL), periods=144, freq="10min").tz_convert("UTC")
    fuertes = {"S1": ["15:10", "15:20", "15:40", "16:30"], "S2": ["14:00", "14:40", "15:20", "16:00", "16:40"]}
    filas = []
    for est, horas in fuertes.items():
        f = {loc(h) for h in horas}
        filas += [(est, t, 25.0 if t in f else 5.0) for t in grilla]
    intensidad = pd.DataFrame(filas, columns=["codigoestacion", "timestamp", "int_t"])
    # etiquetas por la regla de la ventana, calculadas sin usar el módulo
    fuerte = intensidad.pivot(index="timestamp", columns="codigoestacion", values="int_t") >= 20
    y = pd.DataFrame(False, index=fuerte.index, columns=fuerte.columns)
    for k in range(13, 19):                     # τ = t + 130 min ... t + 180 min
        y |= fuerte.shift(-k, fill_value=False)
    pares = y.stack().rename("y").reset_index()
    pares["en_curso"] = fuerte.stack().to_numpy()
    pares = pares[~pares["en_curso"]].drop(columns="en_curso").reset_index(drop=True)
    pares["y"] = pares["y"].astype(int)
    alertas = {("S1", h) for h in ["09:00", "09:20", "11:50", "12:10", "12:30"]} | {("S2", "14:10")}
    alertas = {(e, loc(h)) for e, h in alertas}
    pares["p"] = [0.9 if (e, t) in alertas else 0.1 for e, t in zip(pares.codigoestacion, pares.timestamp)]
    return intensidad, pares, loc

intensidad, pares_m, loc = caso_a_mano()
umbrales_m = pd.Series({"S1": 20.0, "S2": 20.0})
episodios_m, fuertes_m = ev.episodios_lluvia(intensidad, umbrales_m)
print(episodios_m.assign(inicio=episodios_m.inicio.dt.tz_convert(ev.ZONA_LOCAL).dt.strftime("%H:%M"),
                         fin=episodios_m.fin.dt.tz_convert(ev.ZONA_LOCAL).dt.strftime("%H:%M")))
verificar(len(episodios_m) == 3, "3 episodios: E1 (S1, pausa de 10 min unida), E2 (S1, tras 40 min) y E3 (S2, pausas de 30 min unidas)")
verificar(list(episodios_m.inicio) == [loc("15:10"), loc("16:30"), loc("14:00")], "inicios 15:10, 16:30 y 14:00")

   id_episodio codigoestacion inicio    fin
0            0             S1  15:10  15:40
1            1             S1  16:30  16:30
2            2             S2  14:00  16:40
OK   3 episodios: E1 (S1, pausa de 10 min unida), E2 (S1, tras 40 min) y E3 (S2, pausas de 30 min unidas)
OK   inicios 15:10, 16:30 y 14:00


In [ ]:
pares_m["id_episodio"], n_dist = ev.asignar_episodio(pares_m, fuertes_m)
verificar(((pares_m.id_episodio >= 0) == (pares_m.y == 1)).all(), "cada par positivo tiene episodio y ningún negativo lo tiene (coincide con la regla de la etiqueta)")
fila = (pares_m.codigoestacion == "S1") & (pares_m.timestamp == loc("13:30"))
verificar(pares_m.loc[fila, "id_episodio"].item() == 0 and n_dist[fila.to_numpy()].item() == 2,
          "el par de S1 a las 13:30 toca E1 y E2 y queda asignado a E1")

ep_m, av_m = ev.eventos(pares_m, episodios_m, umbral=0.5)
print(ep_m[["id_episodio", "codigoestacion", "avisado", "anticipacion_h"]])
print(av_m.assign(t_primera=av_m.t_primera.dt.tz_convert(ev.ZONA_LOCAL).dt.strftime("%H:%M"))[["codigoestacion", "t_primera", "n_alertas", "tipo"]])
verificar(list(ep_m.avisado) == [True, False, False], "E1 avisado; E2 y E3 no (la alerta de E3 es posterior a su inicio)")
verificar(np.isclose(ep_m.loc[0, "anticipacion_h"], 3.0), "anticipación de E1 = 3.0 h, desde la primera alerta válida (12:10), no desde las 11:50")
verificar(list(av_m.tipo) == ["falso", "correcto", "tardío"], "avisos: falso, correcto (empieza fuera de la ventana) y tardío")
m = ev.metricas_evento(3, 1, 3, 1, 1)
verificar(np.isclose(m["pod_evento"], 1 / 3) and np.isclose(m["far_evento"], 1 / 3), "POD y FAR por evento = 1/3")

OK   cada par positivo tiene episodio y ningún negativo lo tiene (coincide con la regla de la etiqueta)
OK   el par de S1 a las 13:30 toca E1 y E2 y queda asignado a E1
   id_episodio codigoestacion  avisado  anticipacion_h
0            0             S1     True             3.0
1            1             S1    False             NaN
2            2             S2    False             NaN
  codigoestacion t_primera  n_alertas      tipo
0             S1     09:00          2     falso
1             S1     11:50          3  correcto
2             S2     14:10          1    tardío
OK   E1 avisado; E2 y E3 no (la alerta de E3 es posterior a su inicio)
OK   anticipación de E1 = 3.0 h, desde la primera alerta válida (12:10), no desde las 11:50
OK   avisos: falso, correcto (empieza fuera de la ventana) y tardío
OK   POD y FAR por evento = 1/3


## 7. Conjunto sintético con episodios

Las pruebas de estratos necesitan muchos episodios con la estructura real: 20 estaciones, 40 días pluviométricos, episodios de duración variable (con pausas) a cualquier hora, etiquetas calculadas con la regla de la ventana, y exclusión de los pares con lluvia fuerte en curso y con tormenta inminente, como en el índice real. Dos "modelos" con distinto ruido sobre la misma verdad sirven para comprobar que la clasificación de episodios en estratos no depende del modelo.

In [ ]:
def conjunto_eventos(n_est=20, n_dias=40, semilla=11):
    rng = np.random.default_rng(semilla)
    grilla = pd.date_range(pd.Timestamp("2025-01-01 07:00", tz=ev.ZONA_LOCAL), periods=n_dias * 144, freq="10min").tz_convert("UTC")
    n = len(grilla)
    est = [f"E{i:02d}" for i in range(n_est)]
    fuerte = np.zeros((n, n_est), dtype=bool)
    prob_dia = rng.uniform(0.05, 0.5, n_est)          # estaciones con pocos y con muchos episodios
    for j in range(n_est):
        for d in range(n_dias):
            if rng.random() < prob_dia[j]:
                i0 = d * 144 + rng.integers(0, 140)
                for k in range(rng.integers(1, 6)):
                    i = i0 + k * rng.integers(1, 5)
                    if i < n:
                        fuerte[i, j] = True
    F = pd.DataFrame(fuerte, index=grilla, columns=est)
    y = pd.DataFrame(False, index=grilla, columns=est)
    for k in range(13, 19):
        y |= F.shift(-k, fill_value=False)
    inminente = pd.DataFrame(False, index=grilla, columns=est)
    for k in range(1, 13):
        inminente |= F.shift(-k, fill_value=False)
    inminente &= ~y
    intensidad = F.stack().rename("fuerte").reset_index()
    intensidad.columns = ["timestamp", "codigoestacion", "fuerte"]
    intensidad["int_t"] = np.where(intensidad.fuerte, 30.0, 2.0)
    pares = pd.DataFrame({"timestamp": np.repeat(grilla, n_est), "codigoestacion": np.tile(est, n),
                          "y": y.to_numpy().ravel().astype(int),
                          "excluir": (F.to_numpy() | inminente.to_numpy()).ravel()})
    pares = pares[~pares.excluir].drop(columns="excluir").reset_index(drop=True)
    lag = rng.normal(0, 1.0, len(pares))
    pares["p_A"] = 1 / (1 + np.exp(-(-3.0 + 2.0 * pares.y + lag)))
    pares["p_B"] = 1 / (1 + np.exp(-(-3.0 + 2.0 * pares.y + rng.normal(0, 1.5, len(pares)))))
    pares["p10_c13"] = np.where(pares.y == 1, rng.normal(230, 15, len(pares)), rng.normal(260, 15, len(pares)))
    umbrales = pd.Series(rng.uniform(10, 40, n_est), index=est)
    return intensidad[["codigoestacion", "timestamp", "int_t"]], pares, umbrales

intensidad_s, pares_s, umbrales_s = conjunto_eventos()
episodios_s, fuertes_s = ev.episodios_lluvia(intensidad_s, pd.Series(20.0, index=umbrales_s.index))
pares_s["id_episodio"], n_dist_s = ev.asignar_episodio(pares_s, fuertes_s)
print(f"{len(pares_s):,} pares, {pares_s.y.sum():,} positivos, {len(episodios_s)} episodios, "
      f"{(n_dist_s > 1).sum()} pares que tocan dos episodios")
verificar(((pares_s.id_episodio >= 0) == (pares_s.y == 1)).all(), "asignación de episodios coincide con las etiquetas en todo el conjunto")

112,315 pares, 2,368 positivos, 276 episodios, 24 pares que tocan dos episodios
OK   asignación de episodios coincide con las etiquetas en todo el conjunto


### 7.1 Estratos: asignación

Noche: 18:50 es día, 19:00 es noche, 06:50 es noche, 07:00 es día (hora local). Temporada por mes local: el 31-may a las 20:00 local ya es 1-jun en UTC, pero sigue siendo MAM. Terciles: con 61 umbrales distintos, los grupos quedan de 20, 20 y 21 estaciones (o la partición más pareja posible).

In [ ]:
ts = pd.to_datetime(["2025-03-10 18:50", "2025-03-10 19:00", "2025-03-11 06:50", "2025-03-11 07:00"]).tz_localize(ev.ZONA_LOCAL).tz_convert("UTC")
verificar(list(ev.es_noche(ts)) == [False, True, True, False], "noche = 19:00 a 06:59 local")
verificar(list(ev.temporada(pd.to_datetime(["2025-06-01 01:00"]).tz_localize("UTC"))) == ["MAM"], "temporada por mes local (31-may 20:00 local es MAM)")
u61 = pd.Series(np.random.default_rng(0).uniform(10, 40, 61))
grupos, cortes = ev.terciles_umbral(u61)
conteo = grupos.value_counts()
verificar(conteo.max() - conteo.min() <= 1, f"terciles de 61 estaciones: {conteo.to_dict()}")

OK   noche = 19:00 a 06:59 local
OK   temporada por mes local (31-may 20:00 local es MAM)
OK   terciles de 61 estaciones: {'bajo': 21, 'medio': 20, 'alto': 20}


### 7.2 Los estratos reproducen el total

Con cualquier partición, la suma de los estratos debe dar exactamente el total: pares, tormentas, alertas, episodios, avisados, avisos y falsos. Si algún par, episodio o aviso quedara fuera de todo estrato o en dos a la vez, esta prueba falla. Se usa la partición día/noche.

In [ ]:
umbral_s = 0.5
ep_s, av_s = ev.eventos(pares_s.assign(p=pares_s.p_A), episodios_s, umbral_s)
bl_s, _ = ev.tablas_por_bloque(pares_s.timestamp, pares_s.y, pares_s.p_A, umbral_s)
pesos_s = ev.pesos_bootstrap(len(bl_s), semilla=42)
args = dict(pares=pares_s.assign(p=pares_s.p_A), ep=ep_s, avisos=av_s, umbral=umbral_s, pesos=pesos_s, bloques=bl_s)
total = ev.evaluar_por_estrato(**args, estrato_pares=np.full(len(pares_s), "todo"),
                               estrato_ep=np.full(len(ep_s), "todo"), estrato_avisos=np.full(len(av_s), "todo"))
noche = ev.evaluar_por_estrato(**args, estrato_pares=ev.es_noche(pares_s.timestamp),
                               estrato_ep=ev.es_noche(ep_s.inicio), estrato_avisos=ev.es_noche(av_s.t_primera))
cols = ["n_pares", "n_tormentas", "n_alertas", "n_episodios", "n_avisados", "n_avisos", "n_falsos", "n_tardios"]
print(pd.concat([total, noche])[["estrato"] + cols + ["csi", "pod_evento", "far_evento"]].to_string(index=False))
verificar((noche[cols].sum() == total[cols].iloc[0]).all(), "día + noche = total en todos los conteos")
verificar(np.isclose(total.pod_evento.item(), ep_s.avisado.mean()), "POD por evento del total = fracción de episodios avisados")
verificar(0 < total.pod_evento.item() < 1, f"el modelo sintético avisa algunos episodios y no otros (POD por evento = {total.pod_evento.item():.2f})")

estrato  n_pares  n_tormentas  n_alertas  n_episodios  n_avisados  n_avisos  n_falsos  n_tardios      csi         pod_evento          far_evento
   todo   112315         2368        504          275         198       350       135          0 0.146507               0.72 0.38571428571428573
  False    56112         1298        285          150         109       194        79          0 0.148766 0.7266666666666667  0.4072164948453608
   True    56203         1070        219          125          89       156        56          0 0.143744              0.712   0.358974358974359
OK   día + noche = total en todos los conteos
OK   POD por evento del total = fracción de episodios avisados
OK   el modelo sintético avisa algunos episodios y no otros (POD por evento = 0.72)


### 7.3 Nube fría en episodios: momento fijo, independiente del modelo

Cada episodio se clasifica por la condición de nube en su primer par positivo. Dos modelos distintos sobre la misma verdad deben producir exactamente la misma clasificación de episodios; si la regla dependiera de las alertas (por ejemplo, la condición en la primera alerta), los grupos cambiarían de un modelo a otro y la comparación no sería justa.

In [ ]:
p10 = pares_s.set_index(["codigoestacion", "timestamp"])["p10_c13"]
clasif = {}
for nombre in ["p_A", "p_B"]:
    ep_x, _ = ev.eventos(pares_s.assign(p=pares_s[nombre]), episodios_s, umbral_s)
    clave = pd.MultiIndex.from_arrays([ep_x.codigoestacion, ep_x.t_primer_par])
    clasif[nombre] = pd.Series(ev.nube_fria(p10.reindex(clave).to_numpy()), index=ep_x.id_episodio).sort_index()
verificar(clasif["p_A"].equals(clasif["p_B"]), f"misma clasificación de los {len(clasif['p_A'])} episodios con los dos modelos")

OK   misma clasificación de los 275 episodios con los dos modelos


### 7.4 Por estación: mínimo de 10 episodios

Con la estación como estrato y `min_episodios = 10`, las estaciones con menos de 10 episodios evaluables quedan marcadas como insuficientes y sin métricas, pero con sus conteos.

In [ ]:
est = ev.evaluar_por_estrato(**args, estrato_pares=pares_s.codigoestacion.to_numpy(),
                             estrato_ep=ep_s.codigoestacion.to_numpy(), estrato_avisos=av_s.codigoestacion.to_numpy(),
                             min_episodios=ev.MIN_EPISODIOS_ESTACION)
print(est[["estrato", "n_episodios", "insuficiente", "pod_evento", "csi"]].to_string(index=False))
verificar(est.insuficiente.any() and (~est.insuficiente).any(), f"hay estaciones de los dos tipos ({est.insuficiente.sum()} insuficientes)")
verificar((est.insuficiente == (est.n_episodios < 10)).all(), "insuficiente ⇔ menos de 10 episodios")
verificar(est.loc[est.insuficiente, "csi"].isna().all(), "las estaciones insuficientes no tienen métricas")
verificar(est.loc[~est.insuficiente, "csi"].notna().all(), "las demás sí")

estrato  n_episodios  insuficiente         pod_evento      csi
    E00           10         False                0.6 0.152778
    E01           15         False                0.6 0.153846
    E02           29         False 0.8275862068965517 0.167939
    E03            2          True                NaN      NaN
    E04            8          True                NaN      NaN
    E05           19         False 0.5263157894736842 0.111732
    E06            1          True                NaN      NaN
    E07            4          True                NaN      NaN
    E08           15         False 0.7333333333333333 0.145570
    E09           17         False 0.7647058823529411 0.167785
    E10            7          True                NaN      NaN
    E11           20         False                0.8 0.160428
    E12           20         False                0.7 0.130178
    E13           11         False 0.6363636363636364 0.146067
    E14            6          True                NaN  

### 7.5 Anticipación

La anticipación de un episodio avisado está entre 10 min (alerta justo antes del inicio) y 3 h (alerta en el primer par posible), en múltiplos de 10 min.

In [ ]:
a = ep_s.loc[ep_s.avisado, "anticipacion_h"]
print("  ", ev.resumen_anticipacion(ep_s))
verificar(a.between(1 / 6, 3).all(), f"anticipación entre 10 min y 3 h (mín {a.min():.2f} h, máx {a.max():.2f} h)")
verificar(np.allclose((a * 6) % 1, 0) | np.allclose((a * 6) % 1, 1), "anticipación en múltiplos de 10 min")

   {'n_avisados': 198, 'mediana_h': 2.6666666666666665, 'q1_h': 2.3333333333333335, 'q3_h': 2.8333333333333335, 'frac_1h': 1.0}
OK   anticipación entre 10 min y 3 h (mín 1.17 h, máx 3.00 h)
OK   anticipación en múltiplos de 10 min


## 8. Episodios con datos reales

Se construyen los episodios de validación y prueba con la intensidad `int_t` de las etiquetas y los umbrales de cada estación, y se comprueba que la asignación reproduce exactamente las etiquetas: todo par positivo debe recibir un episodio y ningún negativo. Si esta prueba falla, la ventana usada aquí, (t + 2 h, t + 3 h], o la intensidad de la grilla de 10 min no coinciden con las del etiquetado, y las métricas por evento no serían válidas. También se cuentan los pares que tocan dos episodios y la cantidad de episodios por conjunto.

Los umbrales son los de `umbral_operativo_por_estacion.csv` (columna `umbral_mm_h`), los mismos que usa `05-ground_truth_labels` para etiquetar. La intensidad `int_t` del parquet solo cubre los instantes que tienen etiqueta, que terminan 3 h antes del último registro de cada estación; los pares cuya ventana pasa de ese punto no se pueden verificar y se cuentan aparte.

In [ ]:
umbrales_csv = pd.read_csv(RAIZ / "data_ingestion" / "ground_truth" / "umbral_operativo_por_estacion.csv",
                           dtype={"codigoestacion": str})
umbrales_r = umbrales_csv.set_index("codigoestacion")["umbral_mm_h"]
print(umbrales_csv["origen"].value_counts())

lab = etiquetas[etiquetas[COL_ESTACION].astype(str) != NUEVA_GENERACION].copy()
lab[COL_ESTACION] = lab[COL_ESTACION].astype(str)
umbrales_r = umbrales_r[umbrales_r.index.isin(lab[COL_ESTACION].unique())]
print(f"{len(umbrales_r)} estaciones con umbral, {lab[COL_ESTACION].nunique()} en las etiquetas")

episodios_r, fuertes_r = ev.episodios_lluvia(lab, umbrales_r, col_estacion=COL_ESTACION)
pares_r = lab.dropna(subset=["etiqueta"])
pares_r = pares_r[~pares_r.en_curso & ~pares_r.inminente & (pares_r.timestamp >= INICIO_VAL)].copy()
pares_r["id_episodio"], n_dist_r = ev.asignar_episodio(pares_r, fuertes_r, col_estacion=COL_ESTACION)

ultimo = lab.groupby(COL_ESTACION)["timestamp"].max()
verificable = (pares_r["timestamp"] + ev.VENTANA_HASTA <= pares_r[COL_ESTACION].map(ultimo)).to_numpy()
pos = (pares_r.etiqueta == 1).to_numpy()
sin_ep = (pos & verificable & (pares_r.id_episodio < 0).to_numpy()).sum()
neg_con_ep = (~pos & verificable & (pares_r.id_episodio >= 0).to_numpy()).sum()
print(f"pares no verificables (ventana más allá de la intensidad disponible): {(~verificable).sum():,} "
      f"({(pos & ~verificable).sum()} positivos)")
print(f"positivos sin episodio: {sin_ep:,} de {(pos & verificable).sum():,}; "
      f"negativos con episodio: {neg_con_ep:,} de {(~pos & verificable).sum():,}")
verificar(sin_ep == 0 and neg_con_ep == 0, "la asignación de episodios reproduce exactamente las etiquetas")

origen
propio                             52
respaldo pooled                    10
propio (diagnóstico, excepción)     1
Name: count, dtype: int64
62 estaciones con umbral, 62 en las etiquetas
pares no verificables (ventana más allá de la intensidad disponible): 1,115 (0 positivos)
positivos sin episodio: 0 de 13,991; negativos con episodio: 0 de 2,729,859
OK   la asignación de episodios reproduce exactamente las etiquetas


In [ ]:
conjunto = np.where(pares_r.timestamp < pd.Timestamp("2025-01-01", tz=ev.ZONA_LOCAL), "val", "test")
ep_conj = pd.DataFrame({"id_episodio": pares_r.id_episodio, "conjunto": conjunto})[pos].drop_duplicates("id_episodio")
print(f"pares positivos que tocan dos episodios: {(n_dist_r[pos] > 1).sum():,} de {pos.sum():,}")
print(ep_conj.conjunto.value_counts().rename("episodios evaluables (antes de filtrar por L_max = 4)"))

pares positivos que tocan dos episodios: 5 de 13,991
conjunto
test    1114
val      149
Name: episodios evaluables (antes de filtrar por L_max = 4), dtype: int64
